# AnimationStudio - Phases 9-12: Render -> Post-Production -> Publish -> Orchestrate

A single episode end-to-end through the last four studio phases.

- **Phase 9 (Render):** the EpisodeWorkflowFactory's render queue is driven
  from a Phase 7 plan, every shot clip is "rendered" (mock), then the
  regeneration loop is exercised for kept/rejected clips.
- **Phase 10 (Post-Production):** scenes are assembled into a master
  timeline, QC validates the cut, and export presets are listed.
- **Phase 11 (Publishing):** metadata is generated for the episode, a
  publish record is created, and a release schedule is assigned.
- **Phase 12 (Orchestration):** the studio orchestrator runs every earlier
  step (`story`, `storyboard`, `images`, `animation`, `edit`, `qc`,
  `publish`, `monitor`) as a workflow and reports completion.

## Parts

- **Part A (mock, always runs, offline):** the full chain above with
  in-process mocks.  No GPU, no ComfyUI, no network.
- **Part B (real, GPU):** duplicate of the Phase 8 ComfyUI install /
  fp8 Flux download / server start.  Set `RUN_REAL_RENDER = True` when you
  have a GPU runtime AND the Phase 1-3 libraries.  Cell 10 starts the real
  render of one scene through the actual `RenderPipeline`.


In [ ]:
#@title 1. Settings

import os
import subprocess
import sys

# GitHub clone URL for this studio (colab-gpu is the only supported branch).
REPO_URL = "https://github.com/YOUR_ORG/AnimationStudio.git"  #@param {type:"string"}

# colab-gpu -> fp8 Flux (16GB VRAM, best on T4).  master is deprecated/unused.
BRANCH = "colab-gpu"  #@param ["colab-gpu"]

COMFYUI_PORT = 8188  #@param {type:"integer"}
UI_PORT = 8000  #@param {type:"integer"}

# The ONLY thing stored on Google Drive (free tier = 5 GB): the asset DB.
# Shortlisted/approved state survives session resets here.
DRIVE_ROOT = "/content/drive/MyDrive/AnimationStudio"  #@param {type:"string"}
DB = f"{DRIVE_ROOT}/catalog.db"

# The ~17.25 GB model cache. Free Drive cannot hold it -> keep on the Colab
# disk (re-downloaded after a VM reset).  Set True only if you have space.
CACHE_MODELS_IN_DRIVE = False  #@param {type:"boolean"}

# ---- Episode to run through Phases 9-12 ----
SEASON = 1  #@param {type:"integer"}
EPISODE_NUMBER = 1  #@param {type:"integer"}

# ----- Phase 10 post-production knobs -----
GENERATE_METADATA = True  #@param {type:"boolean"}
SCHEDULE_PUBLISH = True  #@param {type:"boolean"}

# ---- Part B (real ComfyUI render of ONE scene) ----
# Part B needs: (a) a GPU runtime, (b) the fp8 Flux model, and (c) the
# character/world libraries already locked.  Keep False for pure Part A.
REAL_SCENE_INDEX = 0  #@param {type:"integer"}
REAL_SHOTS = 3  #@param {type:"integer"}
RUN_REAL_RENDER = False  #@param {type:"boolean"}

START_TUNNEL = True  #@param {type:"boolean"}

# Sync exported/approved output back to GitHub.  Off -> download a zip.
SYNC_TO_GITHUB = True  #@param {type:"boolean"}
GIT_NAME = "Colab Studio"  #@param {type:"string"}
GIT_EMAIL = "colab@animationstudio.local"  #@param {type:"string"}
GITHUB_TOKEN = ""  #@param {type:"string"}

WORK = "/content"
REPO = f"{WORK}/AnimationStudio"
COMFY = f"{WORK}/comfyui"

if REPO_URL.startswith("https://github.com/YOUR_ORG/"):
    raise SystemExit("Set REPO_URL in Cell 1 to your GitHub repository before running.")


In [ ]:
#@title 2. Mount Google Drive (catalog.db only)

from google.colab import drive

drive.mount("/content/drive", force_remount=False)
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("Drive ready (holds catalog.db only):", DRIVE_ROOT)


In [ ]:
#@title 3. Clone repo and install the studio

def run(cmd, **kw):
    print("+ " + " ".join(cmd))
    return subprocess.run(cmd, check=True, **kw)


os.chdir(WORK)
if not os.path.isdir(REPO):
    run(["git", "clone", "--branch", BRANCH, REPO_URL, "AnimationStudio"])
os.chdir(REPO)
run(["git", "checkout", BRANCH])
run(["git", "pull", "origin", BRANCH])

# Studio first (torch is already preinstalled on Colab), then light deps.
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
run([sys.executable, "-m", "pip", "install", "-q",
     "fastapi", "uvicorn", "jinja2", "aiosqlite", "python-multipart",
     "pydantic", "scikit-learn", "imageio", "opencv-python-headless"])
print("Studio installed (branch:", BRANCH, ")")


In [ ]:
#@title 4. Phase 7 upstream: build the episode + per-shot prompts (mock)

# Reuses the Phase 7 planning chain entirely in-process, so Phases 9-12 run
# offline with no GPU, no ComfyUI, and no network.

import sys
sys.path.insert(0, REPO)

from src.story_engine.generator import EpisodeGenerator
from src.production.blueprint_adapter import blueprint_to_episode
from src.production.pipeline import ProductionPipeline

gen = EpisodeGenerator(catalog_path=DB)
bp = gen.generate_episode(season=SEASON, episode_number=EPISODE_NUMBER)
ep = blueprint_to_episode(bp)
pipeline = ProductionPipeline()
prompts = pipeline.generate_prompts(ep)

print(f"episode {ep.id}: '{ep.title}' - {ep.scene_count} scenes, "
      f"{ep.shot_count} shots")
print("Phase 7 upstream plan ready.")


In [ ]:
#@title 5. [PHASE 9] Render queue + clip regeneration workflow (mock)

# Drives the animation RenderPipeline over every shot clip, marks each job
# COMPLETED (mock), then exercises the regeneration loop on a REJECTED clip.

from src.animation.render import RenderPipeline, RenderStatus
from src.animation.regeneration import ClipRegenerationEngine

render = RenderPipeline()
render.submit_batch(list(prompts.keys()))
job_ids = []
while render.queue.pending_count():
    job = render.process_next()
    if job:
        job_ids.append(job.job_id)
        render.complete_job(job.job_id)

print("=" * 72)
print("  PHASE 9 - RENDER QUEUE")
print("=" * 72)
print(f"  submitted + completed: {len(job_ids)} render job(s)")
for job_id in job_ids[:5]:
    job = render.queue.get_job(job_id)
    print(f"    {job.job_id:20s} {job.status.value}")

# Regeneration workflow: reject the first clip and request a re-render.
regen = ClipRegenerationEngine()
first = job_ids[0]
reason = regen.list_reasons()[0]
req = regen.request_regeneration(clip_id=first, reason=reason, notes="motion too fast")
summary = regen.summarize(clip_id=first, reason=reason, notes="")
print("=" * 72)
print("  PHASE 9 - REGENERATION")
print("=" * 72)
print(f"  request: {req.reason} for {first}")
print(f"  summary: {summary}")
print()
print("Phase 9 complete - all clips rendered (mock), regeneration looped.")


In [ ]:
#@title 6. [PHASE 10] Post-production: assemble timeline + QC + export presets

# Builds a SceneAssembly from the episode, assembles a master timeline,
# runs the full PostProductionQC battery, and lists export presets.

from src.story_engine.generator import EpisodeGenerator
from src.production.blueprint_adapter import blueprint_to_episode
from src.post_production import (
    ClipReference, EditingEngine, ExportEngine, PostProductionQC, SceneAssembly,
)

gen = EpisodeGenerator(catalog_path=DB)
bp = gen.generate_episode(season=SEASON, episode_number=EPISODE_NUMBER)
ep = blueprint_to_episode(bp)

sections = {
    "opening": [], "introduction": [], "learning": [], "song": [],
    "practice": [], "review": [], "celebration": [], "outro": [],
}
for scene in ep.scenes:
    section = str(getattr(scene, "type", None) or "learning").lower()
    for shot in scene.shots:
        sections.get(section, sections["learning"]).append(ClipReference(
            clip_id=shot.id, episode=ep.id, scene=scene.id, shot=shot.id,
            duration=max(1.0, getattr(shot, "duration_seconds", 2.0) or 2.0),
        ))

assembly = SceneAssembly(episode_id=ep.id, **sections)
edit = EditingEngine()
timeline = edit.assemble_scenes(ep.id, assembly, ep.title)
qc = PostProductionQC()
result = qc.validate_timeline(timeline)
exports = ExportEngine()

print("=" * 72)
print("  PHASE 10 - POST-PRODUCTION")
print("=" * 72)
print(f"  timeline: {timeline.episode_id}  duration: {timeline.duration_seconds:.1f}s")
print(f"  QC passed: {result.passed}  |  score: {result.score:.1f}")
for check, ok in result.checks.items():
    print(f"    {check:24s} {'OK' if ok else 'FAIL'}")
print(f"  export presets: {', '.join(exports.list_presets().keys())}")
print()
print("Phase 10 complete - the cut is QC-clean and export-ready.")


In [ ]:
#@title 7. [PHASE 11] Publishing: metadata + publish record + schedule

# Generates the episode's metadata/description, creates a publish record,
# validates it is publish-ready, and assigns a release schedule.

from src.story_engine.generator import EpisodeGenerator
from src.production.blueprint_adapter import blueprint_to_episode
from src.publishing.publishing import PublishingEngine
from src.publishing.schedule import SchedulingEngine

gen = EpisodeGenerator(catalog_path=DB)
bp = gen.generate_episode(season=SEASON, episode_number=EPISODE_NUMBER)
ep = blueprint_to_episode(bp)

pub = PublishingEngine()
pkg = pub.prepare_publish_package(
    episode_id=ep.id,
    topic=bp.curriculum_tags[0] if bp.curriculum_tags else "learning",
    character=bp.main_character,
    objective=bp.learning_objective,
    series=bp.season_name or "AI Nursery Studio",
    age_group=str(bp.target_age),
    keywords=bp.keywords,
)
rec = pub.create_record(episode_id=ep.id, channel_id="main",
                        title=pkg["selected_title"])

scheduler = SchedulingEngine()
sched = None
if SCHEDULE_PUBLISH:
    sched = scheduler.create_schedule(episode_id=ep.id, publish_date="2026-09-15")

print("=" * 72)
print("  PHASE 11 - PUBLISHING")
print("=" * 72)
print(f"  selected title : {pkg['selected_title']}")
print(f"  description    : {pkg['description'][:70]}...")
print(f"  record         : {rec.record_id} (session {rec.__class__.__name__})")
if sched is not None:
    print(f"  schedule       : {sched.schedule_id} @ {sched.publish_date} {sched.publish_time} UTC")
print()
print("Phase 11 complete - episode has metadata, a record, and a release slot.")


In [ ]:
#@title 8. [PHASE 12] Studio orchestration: run the full episode workflow

# The studio orchestrator re-drives the complete 8-step production workflow
# (story -> storyboard -> images -> animation -> edit -> qc -> publish ->
# monitor) through the WorkflowEngine, reporting per-step completion.

from src.story_engine.generator import EpisodeGenerator
from src.production.blueprint_adapter import blueprint_to_episode
from src.studio.orchestrator import PipelineOrchestrator

gen = EpisodeGenerator(catalog_path=DB)
bp = gen.generate_episode(season=SEASON, episode_number=EPISODE_NUMBER)
ep = blueprint_to_episode(bp)

orch = PipelineOrchestrator()
orch.setup_defaults()
orch.create_pipeline(ep.id)
completed = orch.process_pipeline(ep.id, passes=20)

print("=" * 72)
print("  PHASE 12 - ORCHESTRATION")
print("=" * 72)
print(f"  pipeline completed: {completed}")
print()
print("Phase 12 complete - the orchestrated production finished.")


In [ ]:
#@title 9. Write the episode production report

from datetime import datetime

report = f"""# Episode Production Report
- Episode: {ep.id}  ({ep.scene_count} scenes / {ep.shot_count} shots)
- Title: {ep.title}
- Phase 9  : {len(job_ids)} render jobs (mock) + regeneration loop
- Phase 10 : timeline {timeline.duration_seconds:.0f}s, QC score {result.score:.0f}/100
- Phase 11 : record {rec.record_id}, title '{pkg['selected_title']}'
- Phase 12 : orchestrator completed = {completed}
- Generated: {datetime.now():%Y-%m-%d %H:%M}
"""
report_path = f"{REPO}/PHASE9_12_EPISODE_REPORT.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)
print("Wrote", report_path)
print(report)


In [ ]:
#@title 10. [PART B] Real ComfyUI render of ONE scene through RenderPipeline

# Requires RUN_REAL_RENDER = True and the ComfyUI setup (Cells 11-13).
# Pushes the scene's shots into the real RenderPipeline and process them.

if RUN_REAL_RENDER:
    import sys
    sys.path.insert(0, f"{REPO}/colab")
    from comfy_helpers import ensure_comfyui_up
    ensure_comfyui_up(port=COMFYUI_PORT, work=WORK, comfy_dir=COMFY)

    from src.generation_engine.comfy_backend import ComfyUIBackend
    from src.generation_engine.base import GenerationInput
    from src.review_ui.combined_repo import check_comfyui

    if not check_comfyui(f"http://localhost:{COMFYUI_PORT}"):
        raise SystemExit("ComfyUI not reachable - start it in Cell 13 first.")

    backend = ComfyUIBackend(server_url=f"http://localhost:{COMFYUI_PORT}")
    scene = ep.scenes[REAL_SCENE_INDEX]
    shot_ids = [s.id for s in scene.shots][:REAL_SHOTS]

    from src.animation.render import RenderPipeline
    real_render = RenderPipeline()
    real_render.submit_batch(shot_ids)
    completed = 0
    while real_render.queue.pending_count():
        job = real_render.process_next()
        if job:
            ok = bool(backend.generate(GenerationInput(
                prompt=prompts[job.clip_id],
                negative_prompt="blurry, distorted, low quality",
                seed=1234, width=1024, height=1024, num_images=1,
            ), asset_type="scene").images)
            if ok:
                real_render.complete_job(job.job_id)
                completed += 1
    print(f"Part B rendered {completed}/{len(shot_ids)} real shot clip(s).")
    os.makedirs(f"{REPO}/Assets/scenes/{ep.id}", exist_ok=True)
else:
    print("RUN_REAL_RENDER is off - skipping Part B (Part A only).")


In [ ]:
#@title 11. Install ComfyUI (Part B prerequisite)

if RUN_REAL_RENDER:
    if not os.path.isdir(COMFY):
        run(["git", "clone", "--depth", "1",
             "https://github.com/comfyanonymous/ComfyUI.git", COMFY])
    run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{COMFY}/requirements.txt"])
    print("ComfyUI ready at", COMFY)
else:
    print("RUN_REAL_RENDER is off - skipping ComfyUI install (Part A only).")


In [ ]:
#@title 12. Download the fp8 Flux model (Part B, Colab disk NOT Drive)

if RUN_REAL_RENDER:
    MODELS = {
        "colab-gpu": {
            "checkpoints/flux1-dev.safetensors":
                "https://huggingface.co/Comfy-Org/flux1-dev/resolve/main/flux1-dev-fp8.safetensors",
        },
    }[BRANCH]

    import shutil

    cache_root = f"{DRIVE_ROOT}/models" if CACHE_MODELS_IN_DRIVE else f"{WORK}/models"
    for rel, url in MODELS.items():
        cached = f"{cache_root}/{rel}"
        link = f"{COMFY}/models/{rel}"
        if not (os.path.exists(cached) and os.path.getsize(cached) > 0):
            os.makedirs(os.path.dirname(cached), exist_ok=True)
            free_gb = shutil.disk_usage("/content").free / 1e9
            if free_gb < 4:
                raise SystemExit(
                    f"Only {free_gb:.1f} GB free on /content - cannot fit the "
                    "~17.25 GB fp8 model. Stop other runtimes or free disk space."
                )
            print(f"Downloading {rel} ...")
            run(["wget", "-q", "-c", "-O", cached, url])
        size_gb = os.path.getsize(cached) / 1e9
        if size_gb < 17.25 * 0.9:
            raise SystemExit(
                f"Model looks truncated: {size_gb:.2f} GB cached - re-run this "
                "cell (wget -c resumes) or delete the file and restart."
            )
        os.makedirs(os.path.dirname(link), exist_ok=True)
        if os.path.lexists(link) and not os.path.islink(link):
            os.remove(link)
        if not os.path.islink(link):
            try:
                os.symlink(cached, link)
            except OSError:
                shutil.copyfile(cached, link)
        print(f"OK {rel} ({size_gb:.2f} GB)")
    print("Model cache:", cache_root)
else:
    print("RUN_REAL_RENDER is off - skipping fp8 Flux download.")


In [ ]:
#@title 13. Start ComfyUI server + verify GPU (Part B)

if RUN_REAL_RENDER:
    import torch
    assert torch.cuda.is_available(), "GPU runtime required for Part B - set Runtime > Change runtime type > T4 GPU."
    print("GPU:", torch.cuda.get_device_name(0))

    import sys
    sys.path.insert(0, f"{REPO}/colab")
    from comfy_helpers import ensure_comfyui_up
    ensure_comfyui_up(port=COMFYUI_PORT, work=WORK, comfy_dir=COMFY)
    print("ComfyUI ready on :" + str(COMFYUI_PORT))
else:
    print("RUN_REAL_RENDER is off - skipping ComfyUI server + GPU check.")


In [ ]:
#@title 14. Launch the Review UI and tunnel (view approved assets)

from src.review_ui.app import create_app
from src.universe.batch_generator import resolve_backend

backend_kind = "comfyui" if RUN_REAL_RENDER else "mock"
app = create_app(
    db_path=DB,
    generation_backend=resolve_backend(backend_kind,
                                       comfyui_url=f"http://localhost:{COMFYUI_PORT}"),
    universe_dir=f"{REPO}/Universe",
    world_dir=f"{REPO}/World",
    assets_dir=f"{REPO}/Assets",
    persist_generated_images=True,
)

import socket
import threading
import uvicorn

ui_alive = False
probe = socket.socket()
probe.settimeout(2)
try:
    probe.connect(("127.0.0.1", UI_PORT))
    ui_alive = True
except Exception:
    ui_alive = False
finally:
    probe.close()

if ui_alive:
    print(f"Review UI already running on :{UI_PORT}")
else:
    config = uvicorn.Config(app, host="0.0.0.0", port=UI_PORT, log_level="warning")
    threading.Thread(target=uvicorn.Server(config).run, daemon=True).start()
    print(f"Review UI starting on :{UI_PORT} ...")

if START_TUNNEL:
    import time
    import re
    import shutil

    if not shutil.which("lt"):
        run(["apt-get", "install", "-y", "-qq", "nodejs", "npm"])
        run(["npm", "install", "-g", "--silent", "localtunnel"])
    lt = shutil.which("lt") or "lt"

    def open_tunnel(port, name):
        out = open(f"{WORK}/{name}.log", "w")
        return subprocess.Popen([lt, "--port", str(port)],
                                stdout=out, stderr=subprocess.STDOUT)

    p1 = open_tunnel(COMFYUI_PORT, "tunnel_comfyui")
    p2 = open_tunnel(UI_PORT, "tunnel_ui")

    urls = {}
    for _ in range(30):
        time.sleep(2)
        for name in ("tunnel_comfyui", "tunnel_ui"):
            txt = open(f"{WORK}/{name}.log").read()
            found = re.findall(r"https://[a-z0-9-]+\.loca\.lt", txt)
            if name not in urls or not urls[name]:
                urls[name] = found
        if urls.get("tunnel_comfyui") and urls.get("tunnel_ui"):
            break

    for name in ("tunnel_comfyui", "tunnel_ui"):
        found = urls.get(name) or []
        print(name, "->", found if found else "no URL yet")
    print("Review UI:", urls.get("tunnel_ui"))
    print("ComfyUI:  ", urls.get("tunnel_comfyui"))


In [ ]:
#@title 15. Sync approved output (GitHub push or manual download)

from datetime import datetime

if SYNC_TO_GITHUB:
    sys.path.insert(0, f"{REPO}/colab")
    from git_sync import auto_sync

    auto_sync(repo=REPO, branch=BRANCH, db_path=DB, token=GITHUB_TOKEN,
              remote_url=REPO_URL,
              git_name=GIT_NAME, git_email=GIT_EMAIL,
              message=f"Phases 9-12 episode report {datetime.now():%Y-%m-%d %H:%M}")
else:
    import zipfile
    from google.colab import files

    zip_path = f"{WORK}/phases9_12_export.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{REPO}/catalog.db", "catalog.db")
        z.write(f"{REPO}/PHASE9_12_EPISODE_REPORT.md", "PHASE9_12_EPISODE_REPORT.md")
        for root, _dirs, names in os.walk(f"{REPO}/Assets"):
            for name in names:
                full = os.path.join(root, name)
                z.write(full, os.path.relpath(full, REPO))
    files.download(zip_path)
    print("Downloaded phases9_12_export.zip (catalog.db + report + Assets).")


## Next steps

- **Part A always ran** - Phases 9, 10, 11, 12 all executed offline against
  the mock render queue, timeline, publishing record, and orchestrator.
- **Part B needs `RUN_REAL_RENDER = True`** plus a GPU runtime and the
  Phase 1-3 libraries.  It pushes ONE scene through a real ComfyUI
  `RenderPipeline` (Cells 11-13 install/download/start first).
- **Approve assets in the Review UI** (Cell 14 tunnel) then re-run Cell 15.
- **The full studio loop** is now covered across the notebooks:
  Phases 1-3 libraries, Phase 7 planning, Phase 8 images, and here the
  render -> post -> publish -> orchestrate chain.

## Troubleshooting

- Part B skipped silently: set `RUN_REAL_RENDER = True` in Cell 1.
- CUDA out of memory: lower `REAL_SHOTS` in Cell 1 or switch to L4/A100.
- Model download stalls: re-run Cell 12 (`wget -c` resumes into the cache).
- QC fails in Cell 6: check `result.checks` - a missing scene section or a
  clip without a duration is the usual cause.
